# 投机解码

补充一些细节

## 抽样标准

```
for each drafted token x_t:
    r ~ Uniform(0, 1)
    if r < p(x_t) / q(x_t):   # true model vs draft model
        accept x_t
    else:
        # + means postive parts
        sample replacement from residual: (p - q)+ / ||(p - q)+||_1
        stop
```

## 树状注意力

EAGLE2 按树状生成了多个候选，使用特殊的树状注意力掩码即可在一次前向过程中验证。

```
            Root
            / \
           a   b
          / \ / \
         c  d e  f

空白部分按自注意力三角掩码
   a  b  c  d  e  f
a  1  
b  0  1
c  1  0  1
d  1  0  0  1
e  0  1  0  0  1
f  0  1  0  0  0  1
```

## 使用/不推荐场景

推荐：
- 容易预测的场景，聊天、代码补全等。输出结构化，接受率高
- GPU不是瓶颈，树状候选，支持一次前向验证一大批。

不推荐：
- 输出随机性大。温度高，创作，接受率低
- 批处理时算力已经不够了，不再能够支持树状校验。
- 小模型不值得使用草案模型

# 动手编码

In [ ]:
import numpy as np
import random


def speculative_decode(
    main_model,
    draft_model,
    context,
    K,
) -> list[int]:
    """Speculative sampling (Leviathan / Chen).

    Convention: p = draft, q = target (main).
    main_model.probs_for_drafts(context, proposed) -> list/array of length K+1,
      where q_list[k] is q(· | context + proposed[:k]), and q_list[K] is for bonus.
    """
    prefix = list(context)
    proposed = []
    p_list = []  # draft distributions

    # 1) autoregressive draft: each step conditions on previous draft tokens
    for _ in range(K):
        token, p = draft_model.sample_token(prefix)
        proposed.append(token)
        p_list.append(p)
        prefix.append(token)

    accepted = []

    # 2) one target forward -> K+1 distributions
    q_list = main_model.probs_for_drafts(context, proposed)

    for k, token in enumerate(proposed):
        q = q_list[k]
        p = p_list[k]
        ratio = q[token] / max(p[token], 1e-12)
        if random.random() < min(1.0, ratio):
            accepted.append(token)
        else:
            residual = np.maximum(q - p, 0)
            residual = residual / np.sum(residual)
            accepted.append(int(np.random.choice(len(residual), p=residual)))
            return accepted

    # 3) all K accepted -> sample bonus from q after full draft
    bonus = int(np.random.choice(len(q_list[K]), p=q_list[K]))
    accepted.append(bonus)
    return accepted



In [ ]:
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap


def tree_attention_mask(parents: list[int]) -> np.ndarray:
    """Build tree attention mask from parent pointers.

    parents[i] = parent index of node i, or -1 if root (of the draft tree).
    mask[i, j] = 1 iff j is on the path from i up to root (including i itself).
    Siblings cannot see each other; only ancestors + self.
    """
    n = len(parents)
    if n == 0:
        return np.zeros((0, 0), dtype=np.int8)

    for i, p in enumerate(parents):
        if p < -1 or p >= n:
            raise ValueError(f"parents[{i}]={p} out of range")
        if p == i:
            raise ValueError(f"parents[{i}] points to itself")

    mask = np.zeros((n, n), dtype=np.int8)
    for i in range(n):
        cur = i
        seen = set()
        while cur != -1:
            if cur in seen:
                raise ValueError(f"cycle detected at node {i}")
            seen.add(cur)
            mask[i, cur] = 1
            cur = parents[cur]
    return mask


def expected_mask_from_notes() -> tuple[np.ndarray, list[int]]:
    """Example tree from the notes:

            (virtual roots)
              a       b
             / \\     / \\
            c   d   e   f

    indices: 0=a, 1=b, 2=c, 3=d, 4=e, 5=f
    """
    parents = [-1, -1, 0, 0, 1, 1]
    return tree_attention_mask(parents), parents


mask, parents = expected_mask_from_notes()
labels = ["a", "b", "c", "d", "e", "f"]
print("parents:", parents)
print("mask:\n", mask)

assert mask[0].tolist() == [1, 0, 0, 0, 0, 0]
assert mask[1].tolist() == [0, 1, 0, 0, 0, 0]
assert mask[2].tolist() == [1, 0, 1, 0, 0, 0]  # c -> a,c
assert mask[5].tolist() == [0, 1, 0, 0, 0, 1]  # f -> b,f
print("matches note example: OK")


In [ ]:
def plot_tree_and_mask(parents: list[int], labels: list[str] | None = None, title: str = "Tree attention"):
    """Visualize draft tree (left) and attention mask heatmap (right)."""
    n = len(parents)
    labels = labels or [str(i) for i in range(n)]
    mask = tree_attention_mask(parents)

    # --- layout: roots at top, children below ---
    children = {i: [] for i in range(n)}
    roots = []
    for i, p in enumerate(parents):
        if p == -1:
            roots.append(i)
        else:
            children[p].append(i)

    pos = {}  # node -> (x, y)

    def layout(node, depth, x_left, x_right):
        kids = children[node]
        pos[node] = ((x_left + x_right) / 2, -depth)
        if not kids:
            return
        span = (x_right - x_left) / len(kids)
        for k, ch in enumerate(kids):
            layout(ch, depth + 1, x_left + k * span, x_left + (k + 1) * span)

    if roots:
        span = 1.0 / len(roots)
        for r_i, r in enumerate(roots):
            layout(r, 0, r_i * span, (r_i + 1) * span)

    fig, axes = plt.subplots(1, 2, figsize=(10, 4))

    # left: tree
    ax = axes[0]
    for i, p in enumerate(parents):
        if p != -1:
            ax.plot([pos[p][0], pos[i][0]], [pos[p][1], pos[i][1]], "k-", lw=1.2)
    for i, (x, y) in pos.items():
        ax.scatter([x], [y], s=600, c="#4C78A8", zorder=3)
        ax.text(x, y, labels[i], ha="center", va="center", color="white", fontsize=11, fontweight="bold")
    ax.set_axis_off()
    ax.set_title("Draft tree (edge = parent→child)")

    # right: mask heatmap
    ax = axes[1]
    cmap = ListedColormap(["#F2F2F2", "#4C78A8"])
    im = ax.imshow(mask, cmap=cmap, vmin=0, vmax=1)
    ax.set_xticks(range(n), labels)
    ax.set_yticks(range(n), labels)
    ax.set_xlabel("key (attend to)")
    ax.set_ylabel("query")
    ax.set_title("Tree attention mask (1 = visible)")
    for i in range(n):
        for j in range(n):
            ax.text(j, i, str(int(mask[i, j])), ha="center", va="center",
                    color="white" if mask[i, j] else "#888", fontsize=9)
    # annotate one path: deepest node
    deepest = max(range(n), key=lambda i: mask[i].sum())
    ax.set_xlabel(f"key (attend to)  |  e.g. query '{labels[deepest]}' sees {[labels[j] for j in range(n) if mask[deepest, j]]}")

    fig.suptitle(title, y=1.02)
    plt.tight_layout()
    plt.show()
    return mask


# note example
plot_tree_and_mask(
    parents=[-1, -1, 0, 0, 1, 1],
    labels=["a", "b", "c", "d", "e", "f"],
    title="Note example: siblings blocked, ancestors visible",
)

# deeper chain for contrast
plot_tree_and_mask(
    parents=[-1, 0, 1, 1, 0],
    labels=["r", "x", "y", "z", "w"],
    title="Deeper path: y sees r→x→y",
)
